# Climate-Driven Vector-Borne Dengue Outbreak Surveillance Engine
### **Project CCHAIN** | *Climate Change, Health, and Artificial Intelligence in the Philippines*
**Laboratory Activity 1: Comprehensive Spatial-Temporal Epidemiological Pipeline**

---

## Executive Overview & Operational Problem Statement

> Local Government Units (LGUs) and City Health Offices across the Philippines face severe operational challenges during seasonal dengue outbreaks. Interventions are often **reactive and delayed**, initiated only after hospital triage beds are overwhelmed. 
>
> By integrating **20 years of downscaled ERA5-Land climate reanalysis**, **DOH/LGU epidemiological surveillance**, **Google Open Buildings footprints**, and **WorldPop human exposure counts**, this pipeline models the **non-linear 30-to-90 day biological lag relationship** between climate anomalies, urban density, and disease outbreaks. This enables municipal health officers to deploy targeted vector control and reallocate hospital capacity **30 to 60 days in advance**.

```
┌─────────────────────────────────────────────────────────────────────────────────────────────┐
│                           4-STAGE ANALYTICS SURVEILLANCE PIPELINE                           │
├───────────────────────────┬─────────────────────────────┬───────────────────────────────────┤
│ Stage                     │ Analytics Type              │ Key Output / Deliverable          │
├───────────────────────────┼─────────────────────────────┼───────────────────────────────────┤
│ 1. Geospatial & Historical│ Descriptive Analytics       │ 20-Year Baseline & Spatial Maps   │
│ 2. Biological Lags & Urban│ Diagnostic Analytics        │ Distributed Lags & Physical Terms │
│ 3. Multi-Horizon ML Model │ Predictive Analytics        │ 30D & 60D Outbreak Classifiers    │
│ 4. Triage & Vector Matrix │ Prescriptive Analytics      │ Automated LGU Action Framework    │
└───────────────────────────┴─────────────────────────────┴───────────────────────────────────┘
```

In [ ]:
# Environment Setup & Library Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import shapely.wkt

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
    fbeta_score,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
    accuracy_score,
    confusion_matrix,
    classification_report
)
import lightgbm as lgb
import xgboost as xgb

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

# Directory Configuration (Handles execution from root, subdirectories, or notebooks/)
candidates_raw = [
    Path.cwd() / 'data' / 'cchain_raw',
    Path.cwd().parent / 'data' / 'cchain_raw',
    Path.cwd().parent.parent / 'datasets' / 'cchain_raw',
    Path.cwd().parent / 'datasets' / 'cchain_raw',
    Path.cwd() / 'datasets' / 'cchain_raw',
    Path('../../datasets/cchain_raw'),
    Path('../datasets/cchain_raw'),
    Path('datasets/cchain_raw'),
    Path('data/cchain_raw'),
]
RAW_DATA_DIR = next((c for c in candidates_raw if (c / 'location.csv').exists()), Path('data/cchain_raw'))
if (Path.cwd() / 'src').exists():
    BASE_DIR = Path.cwd()
elif (Path.cwd() / 'DMA' / 'Climate-Driven Vector-Borne Outbreak Surveillance' / 'src').exists():
    BASE_DIR = Path.cwd() / 'DMA' / 'Climate-Driven Vector-Borne Outbreak Surveillance'
elif (Path.cwd().parent / 'src').exists():
    BASE_DIR = Path.cwd().parent
else:
    BASE_DIR = Path.cwd()
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CITY_CODE = 'PH104305000'  # Cagayan de Oro City
PILOT_CITY_NAME = 'Cagayan de Oro City'
TARGET_DISEASE = 'DENGUE FEVER'

print(f'[+] Pipeline Initialized for: {PILOT_CITY_NAME} ({PILOT_CITY_CODE})')
print(f'[+] Target Path: {RAW_DATA_DIR.resolve()}')

## Step 1: Spatial Master Reference & Contiguity Weights Matrix ($W$)

To model how disease outbreaks diffuse between adjacent urban communities, we parse the boundary geometries (`brgy_geography.csv`) into **Shapely WKT Polygons** and construct a row-normalized **Spatial Contiguity Weights Matrix** ($W$):

$$W_{ij} = \frac{A_{ij}}{\sum_{k} A_{ik}} \quad \text{where } A_{ij} = 1 \text{ if } \text{Barangay}_i \text{ touches/intersects } \text{Barangay}_j, \text{ else } 0$$

In [ ]:
# 1. Load Barangay Master Location Table
df_loc = pd.read_csv(RAW_DATA_DIR / 'location.csv')
cdo_brgys = df_loc[df_loc['adm3_pcode'] == PILOT_CITY_CODE][
    ['adm1_en', 'adm2_en', 'adm3_pcode', 'adm3_en', 'adm4_pcode', 'adm4_en', 'brgy_total_area']
].drop_duplicates(subset=['adm4_pcode']).reset_index(drop=True)

target_pcodes = sorted(cdo_brgys['adm4_pcode'].unique().tolist())
num_brgys = len(target_pcodes)
print(f'[+] Identified {num_brgys} unique barangays in {PILOT_CITY_NAME}.')

# 2. Parse WKT Geometries & Construct Adjacency Matrix
df_geo = pd.read_csv(RAW_DATA_DIR / 'brgy_geography.csv')
cdo_geo = df_geo[df_geo['adm4_pcode'].isin(target_pcodes)].drop_duplicates(subset=['adm4_pcode']).copy()
cdo_geo['poly'] = cdo_geo['geometry'].apply(shapely.wkt.loads)
pcode_to_poly = dict(zip(cdo_geo['adm4_pcode'], cdo_geo['poly']))

adj_matrix = np.zeros((num_brgys, num_brgys), dtype=float)
for i, pcode_i in enumerate(target_pcodes):
    poly_i = pcode_to_poly.get(pcode_i)
    if poly_i is None:
        continue
    for j, pcode_j in enumerate(target_pcodes):
        if i != j:
            poly_j = pcode_to_poly.get(pcode_j)
            if poly_j is not None and (poly_i.touches(poly_j) or poly_i.intersects(poly_j)):
                adj_matrix[i, j] = 1.0

# Row-normalize spatial weights matrix W
row_sums = adj_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
W_spatial = adj_matrix / row_sums

coastal_map = dict(zip(cdo_geo['adm4_pcode'], cdo_geo['brgy_is_coastal'].astype(int)))
cdo_brgys['brgy_is_coastal'] = cdo_brgys['adm4_pcode'].map(coastal_map).fillna(0).astype(int)

print(f'[+] Spatial Contiguity Matrix W: {W_spatial.shape[0]}x{W_spatial.shape[1]}')
print(f'[+] Total Spatial Neighbor Edges: {int(adj_matrix.sum())} (Mean: {adj_matrix.sum()/num_brgys:.2f} neighbors/barangay)')
cdo_brgys.head()

## Step 2: Multi-Source Data Ingestion & Monthly Resampling

We ingest and resample high-resolution health, climate, and structural datasets to a standardized monthly space-time grid:
1. **Health Surveillance:** DOH/LGU disaggregated dengue morbidity records (`disease_lgu_disaggregated_totals.csv`).
2. **Atmospheric Climate:** ERA5-Land daily reanalysis aggregated to monthly total rainfall, mean/max temperature, heat index, and relative humidity (`climate_atmosphere.csv`).
3. **Built-Environment Susceptibility:** Google Open Buildings building counts, density, and built-up area ratios (`google_open_buildings.csv`).
4. **Human Exposure:** WorldPop annual population density records (`worldpop_population.csv`).

In [ ]:
# 1. Ingest Health Surveillance (Dengue Cases & Deaths)
df_lgu = pd.read_csv(RAW_DATA_DIR / 'disease_lgu_disaggregated_totals.csv')
df_dengue = df_lgu[
    (df_lgu['adm3_pcode'] == PILOT_CITY_CODE) &
    (df_lgu['disease_common_name'] == TARGET_DISEASE) &
    (df_lgu['adm4_pcode'].isin(target_pcodes))
].copy()

df_dengue['date'] = pd.to_datetime(df_dengue['date']).dt.to_period('M').dt.to_timestamp()
df_health_agg = df_dengue.groupby(['adm4_pcode', 'date'], as_index=False).agg(
    dengue_cases=('case_total', 'sum'),
    dengue_deaths=('death_total', 'sum')
)

# 2. Ingest Atmospheric Weather Data (ERA5-Land)
df_clim = pd.read_csv(RAW_DATA_DIR / 'climate_atmosphere.csv')
df_clim = df_clim[df_clim['adm4_pcode'].isin(target_pcodes)].copy()
df_clim['date'] = pd.to_datetime(df_clim['date']).dt.to_period('M').dt.to_timestamp()

df_clim_agg = df_clim.groupby(['adm4_pcode', 'date'], as_index=False).agg(
    pr_monthly_total_mm=('pr', 'sum'),
    tave_monthly_mean_c=('tave', 'mean'),
    tmin_monthly_mean_c=('tmin', 'mean'),
    tmax_monthly_mean_c=('tmax', 'mean'),
    heat_index_monthly_mean_c=('heat_index', 'mean'),
    heat_index_monthly_max_c=('heat_index', 'max'),
    rh_monthly_mean_pct=('rh', 'mean'),
    wind_speed_monthly_mean=('wind_speed', 'mean'),
    solar_rad_monthly_mean=('solar_rad', 'mean')
)

# 3. Ingest Built-Environment & Population Data
df_bldgs = pd.read_csv(RAW_DATA_DIR / 'google_open_buildings.csv')
df_bldgs_cdo = df_bldgs[df_bldgs['adm4_pcode'].isin(target_pcodes)][
    ['adm4_pcode', 'google_bldgs_count', 'google_bldgs_density', 'google_bldgs_pct_built_up_area', 'google_bldgs_area_mean']
].drop_duplicates(subset=['adm4_pcode'])

df_pop = pd.read_csv(RAW_DATA_DIR / 'worldpop_population.csv')
df_pop_cdo = df_pop[df_pop['adm4_pcode'].isin(target_pcodes)].sort_values(by='date').groupby('adm4_pcode').last().reset_index()[
    ['adm4_pcode', 'pop_count_total']
]
df_static = df_bldgs_cdo.merge(df_pop_cdo, on='adm4_pcode', how='left')

print(f'[+] Ingested {len(df_health_agg):,} health observations & {len(df_clim_agg):,} monthly climate records.')

## Step 3: Exploratory Data Analysis (EDA) & Biological Visualizations

Before training models, we analyze the **seasonal wave patterns** and **lagged cross-correlations** between rainfall anomalies, ambient thermal stress, and dengue outbreaks.

In [ ]:
# EDA Plot 1: 20-Year Monthly Dengue Caseload vs. Precipitation & Heat Index Trend
city_monthly_health = df_health_agg.groupby('date')['dengue_cases'].sum().reset_index()
city_monthly_clim = df_clim_agg.groupby('date').agg({
    'pr_monthly_total_mm': 'mean',
    'heat_index_monthly_mean_c': 'mean'
}).reset_index()
df_eda_ts = city_monthly_health.merge(city_monthly_clim, on='date', how='inner')

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

# Plot rainfall as background bars
ax1.bar(df_eda_ts['date'], df_eda_ts['pr_monthly_total_mm'], width=20, color='#6baed6', alpha=0.45, label='Monthly Rainfall (mm)')
ax1.set_ylabel('Total Rainfall (mm)', color='#2171b5', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#2171b5')

# Plot Dengue cases as foreground line
ax2.plot(df_eda_ts['date'], df_eda_ts['dengue_cases'], color='#cb181d', linewidth=2.0, label='Reported Dengue Cases')
ax2.set_ylabel('Citywide Dengue Cases', color='#cb181d', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#cb181d')

plt.title('20-Year Historical Trajectory: Monsoon Precipitation vs. Dengue Outbreak Waves (2003-2022)', fontsize=13, fontweight='bold', pad=12)
fig.tight_layout()
plt.show()

In [ ]:
# EDA Plot 2: Cross-Correlation Lag Analysis (Rainfall & Heat Index vs. Dengue Cases)
lags = list(range(0, 7))
rain_corrs = [df_eda_ts['dengue_cases'].corr(df_eda_ts['pr_monthly_total_mm'].shift(lag)) for lag in lags]
heat_corrs = [df_eda_ts['dengue_cases'].corr(df_eda_ts['heat_index_monthly_mean_c'].shift(lag)) for lag in lags]

fig, ax = plt.subplots(figsize=(10, 4.5))
x_idx = np.arange(len(lags))
width = 0.35

ax.bar(x_idx - width/2, rain_corrs, width, label='Precipitation Lag (mm)', color='#3182bd')
ax.bar(x_idx + width/2, heat_corrs, width, label='Heat Index Lag (°C)', color='#de2d26')

ax.set_xlabel('Lag Delay in Months (t - k)', fontweight='bold')
ax.set_ylabel('Pearson Correlation (r)', fontweight='bold')
ax.set_title('Cross-Correlation: Atmospheric Drivers vs. Dengue Morbidity Surge by Lag Month', fontsize=12, fontweight='bold')
ax.set_xticks(x_idx)
ax.set_xticklabels([f'{k} Month(s)' for k in lags])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

print('[i] Peak biological correlation observed at Lag-1m and Lag-2m delays.')

## Step 4: Space-Time Grid Alignment & Advanced Feature Engineering

We engineer 4 categories of epidemiological predictors without future data leakage:
1. **Distributed Meteorological Lags (1m, 2m, 3m, 4m):** Capturing the 4–8 week *Aedes aegypti* vector breeding and Extrinsic Incubation Period (EIP).
2. **Physical Urban-Climate Interactions:**
   - $\text{Runoff Risk} = \text{Built-Up Surface \%} \times \text{Precipitation Lag}$
   - $\text{Urban Heat Trap} = \text{Building Density} \times \text{Heat Index Lag}$
   - $\text{Host Exposure Index} = \text{Population Density} \times \text{Built-Up \%}$
3. **Spatial Contiguity Spillover Lags ($W \times Y$):** Capturing vector and human cross-border contagion from adjacent barangays.
4. **Outbreak Ground Truth Label ($Y$):** Formulated as exceeding the 75th historical percentile per barangay (with a public health floor of 5 cases): 
   $$y_{i,t} = \mathbb{I}\left(\text{Cases}_{i,t} \ge \max(5, P_{75}(\text{Cases}_i))\right)$$

In [ ]:
# 1. Align Space-Time Product Grid
min_date = df_clim_agg['date'].min()
max_date = df_clim_agg['date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='MS')
grid_idx = pd.MultiIndex.from_product([target_pcodes, all_dates], names=['adm4_pcode', 'date']).to_frame().reset_index(drop=True)

df_merged = grid_idx.merge(cdo_brgys, on='adm4_pcode', how='left')
df_merged = df_merged.merge(df_clim_agg, on=['adm4_pcode', 'date'], how='left')
df_merged = df_merged.merge(df_health_agg, on=['adm4_pcode', 'date'], how='left')
df_merged = df_merged.merge(df_static, on='adm4_pcode', how='left')

df_merged['dengue_cases'] = df_merged['dengue_cases'].fillna(0).astype(int)
df_merged['dengue_deaths'] = df_merged['dengue_deaths'].fillna(0).astype(int)
df_merged['pop_density_imputed'] = df_merged['pop_count_total'] / df_merged['brgy_total_area']

# Outbreak Ground Truth Definition (Pre-2019 Training Baseline to Prevent Target Leakage)
train_slice = df_merged[df_merged['date'] < '2019-01-01']
p75_training_thresholds = (
    train_slice.groupby('adm4_pcode')['dengue_cases']
    .quantile(0.75)
    .apply(lambda q: max(5.0, float(q)))
    .to_dict()
)
df_merged['brgy_p75_threshold'] = df_merged['adm4_pcode'].map(p75_training_thresholds).fillna(5.0)
df_merged['is_outbreak'] = (df_merged['dengue_cases'] >= df_merged['brgy_p75_threshold']).astype(int)
df_merged = df_merged.sort_values(by=['adm4_pcode', 'date']).reset_index(drop=True)

# 2. Meteorological Lags & Rolling Metrics
for lag in [1, 2, 3, 4]:
    df_merged[f'pr_total_mm_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['pr_monthly_total_mm'].shift(lag)
    df_merged[f'heat_index_mean_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['heat_index_monthly_mean_c'].shift(lag)
    df_merged[f'tave_mean_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['tave_monthly_mean_c'].shift(lag)
    df_merged[f'rh_mean_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['rh_monthly_mean_pct'].shift(lag)

df_merged['pr_rolling_3m_lag1m'] = (df_merged['pr_total_mm_lag_1m'] + df_merged['pr_total_mm_lag_2m'] + df_merged['pr_total_mm_lag_3m']) / 3.0
df_merged['pr_rolling_3m_lag2m'] = (df_merged['pr_total_mm_lag_2m'] + df_merged['pr_total_mm_lag_3m'] + df_merged['pr_total_mm_lag_4m']) / 3.0

# 3. Physical Urban-Climate Interactions
df_merged['runoff_risk_lag1m'] = df_merged['google_bldgs_pct_built_up_area'] * df_merged['pr_total_mm_lag_1m']
df_merged['runoff_risk_lag2m'] = df_merged['google_bldgs_pct_built_up_area'] * df_merged['pr_total_mm_lag_2m']
df_merged['urban_heat_trap_lag1m'] = df_merged['google_bldgs_density'] * df_merged['heat_index_mean_lag_1m']
df_merged['urban_heat_trap_lag2m'] = df_merged['google_bldgs_density'] * df_merged['heat_index_mean_lag_2m']
df_merged['host_exposure_index'] = df_merged['pop_density_imputed'] * df_merged['google_bldgs_pct_built_up_area']

# 4. Autoregressive & Spatial Contiguity Spillover (W * Y)
for lag in [1, 2]:
    df_merged[f'dengue_cases_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['dengue_cases'].shift(lag)
    df_merged[f'is_outbreak_lag_{lag}m'] = df_merged.groupby('adm4_pcode')['is_outbreak'].shift(lag)

cases_pivot = df_merged.pivot(index='date', columns='adm4_pcode', values='dengue_cases')[target_pcodes]
outbreak_pivot = df_merged.pivot(index='date', columns='adm4_pcode', values='is_outbreak')[target_pcodes]

spatial_cases = pd.DataFrame(np.dot(cases_pivot.values, W_spatial.T), index=cases_pivot.index, columns=target_pcodes)
spatial_outbreak = pd.DataFrame(np.dot(outbreak_pivot.values, W_spatial.T), index=outbreak_pivot.index, columns=target_pcodes)

spatial_cases_long = spatial_cases.reset_index().melt(id_vars='date', var_name='adm4_pcode', value_name='spatial_cases_current')
spatial_outbreak_long = spatial_outbreak.reset_index().melt(id_vars='date', var_name='adm4_pcode', value_name='spatial_outbreak_current')

df_merged = df_merged.merge(spatial_cases_long, on=['adm4_pcode', 'date'], how='left')
df_merged = df_merged.merge(spatial_outbreak_long, on=['adm4_pcode', 'date'], how='left')

df_merged = df_merged.sort_values(by=['adm4_pcode', 'date']).reset_index(drop=True)
df_merged['spatial_lag_cases_1m'] = df_merged.groupby('adm4_pcode')['spatial_cases_current'].shift(1)
df_merged['spatial_lag_cases_2m'] = df_merged.groupby('adm4_pcode')['spatial_cases_current'].shift(2)
df_merged['spatial_lag_outbreak_1m'] = df_merged.groupby('adm4_pcode')['spatial_outbreak_current'].shift(1)
df_merged['spatial_lag_outbreak_2m'] = df_merged.groupby('adm4_pcode')['spatial_outbreak_current'].shift(2)

df_merged = df_merged.drop(columns=['spatial_cases_current', 'spatial_outbreak_current'])
df_final = df_merged.dropna(subset=['pr_total_mm_lag_4m']).copy()

# Export ready matrix
ready_csv = PROCESSED_DATA_DIR / 'cchain_cdo_dengue_surveillance_ready.csv'
df_final.to_csv(ready_csv, index=False)
print(f'[+] Engineered Space-Time Matrix: {df_final.shape[0]:,} rows x {df_final.shape[1]} features.')

## Step 5: Multi-Model Tournament & Early Warning Evaluation

We evaluate across **30-Day Lead ($T+1$)** and **60-Day Lead ($T+2$)** operational horizons using a strict **Out-of-Time Temporal Holdout Split**:
* **Training Partition:** 2003–2018 ($N=15,120$ samples)
* **Testing Partition:** 2019–2022 ($N=3,840$ unseen samples)

To prioritize public health safety (minimizing missed outbreaks), we optimize decision thresholds using the **$F_2$-Score** (weighting recall twice as heavily as precision).

In [ ]:
# Feature Partitioning for Operational Lead Times
FEATURES_30D_HORIZON = [
    'pr_total_mm_lag_1m', 'pr_total_mm_lag_2m', 'pr_total_mm_lag_3m', 'pr_rolling_3m_lag1m',
    'heat_index_mean_lag_1m', 'heat_index_mean_lag_2m', 'heat_index_mean_lag_3m',
    'tave_mean_lag_1m', 'tave_mean_lag_2m', 'tave_mean_lag_3m',
    'rh_mean_lag_1m', 'rh_mean_lag_2m',
    'google_bldgs_count', 'google_bldgs_density', 'google_bldgs_pct_built_up_area', 'google_bldgs_area_mean',
    'brgy_total_area', 'brgy_is_coastal', 'pop_density_imputed',
    'runoff_risk_lag1m', 'runoff_risk_lag2m', 'urban_heat_trap_lag1m', 'host_exposure_index',
    'dengue_cases_lag_1m', 'is_outbreak_lag_1m',
    'spatial_lag_cases_1m', 'spatial_lag_outbreak_1m'
]

FEATURES_60D_HORIZON = [
    'pr_total_mm_lag_2m', 'pr_total_mm_lag_3m', 'pr_total_mm_lag_4m', 'pr_rolling_3m_lag2m',
    'heat_index_mean_lag_2m', 'heat_index_mean_lag_3m', 'heat_index_mean_lag_4m',
    'tave_mean_lag_2m', 'tave_mean_lag_3m', 'tave_mean_lag_4m',
    'rh_mean_lag_2m', 'rh_mean_lag_3m',
    'google_bldgs_count', 'google_bldgs_density', 'google_bldgs_pct_built_up_area', 'google_bldgs_area_mean',
    'brgy_total_area', 'brgy_is_coastal', 'pop_density_imputed',
    'runoff_risk_lag2m', 'urban_heat_trap_lag2m', 'host_exposure_index',
    'dengue_cases_lag_2m', 'is_outbreak_lag_2m',
    'spatial_lag_cases_2m', 'spatial_lag_outbreak_2m'
]

train_mask = df_final['date'] < '2019-01-01'
test_mask = df_final['date'] >= '2019-01-01'

y_train = df_final.loc[train_mask, 'is_outbreak'].values
y_test = df_final.loc[test_mask, 'is_outbreak'].values
pos_weight = (len(y_train) - sum(y_train)) / max(1, sum(y_train))

def optimize_fbeta_threshold(y_true, y_probs, beta=2.0):
    best_th = 0.50
    best_f = 0.0
    for th in np.linspace(0.05, 0.95, 91):
        preds = (y_probs >= th).astype(int)
        score = fbeta_score(y_true, preds, beta=beta, zero_division=0)
        if score > best_f:
            best_f = score
            best_th = th
    return best_th, best_f


In [ ]:
# Benchmark Execution for Both Horizons
results = []
trained_models_dict = {}
predictions_dict = {}
feature_cols_dict = {}

for h_name, feat_cols in [('30-Day Lead (T+1)', FEATURES_30D_HORIZON), ('60-Day Lead (T+2)', FEATURES_60D_HORIZON)]:
    X_train = df_final.loc[train_mask, feat_cols].fillna(0)
    X_test = df_final.loc[test_mask, feat_cols].fillna(0)

    # Fresh model instances per horizon
    horizon_models = {
        'Logistic Regression': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
        ]),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.03, scale_pos_weight=pos_weight,
            random_state=42, verbose=-1
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.03, scale_pos_weight=pos_weight,
            random_state=42, eval_metric='logloss'
        )
    }

    for m_name, model in horizon_models.items():
        model.fit(X_train, y_train)
        key = f'{m_name}_{h_name}'
        trained_models_dict[key] = model
        feature_cols_dict[key] = feat_cols

        probs_test = model.predict_proba(X_test)[:, 1]
        probs_train = model.predict_proba(X_train)[:, 1]
        predictions_dict[key] = probs_test

        opt_th, _ = optimize_fbeta_threshold(y_train, probs_train, beta=2.0)
        preds_opt = (probs_test >= opt_th).astype(int)

        roc_auc = roc_auc_score(y_test, probs_test)
        pr_auc = average_precision_score(y_test, probs_test)
        acc = accuracy_score(y_test, preds_opt)
        rec = recall_score(y_test, preds_opt, zero_division=0)
        prec = precision_score(y_test, preds_opt, zero_division=0)
        f1_opt = f1_score(y_test, preds_opt, zero_division=0)
        f2_opt = fbeta_score(y_test, preds_opt, beta=2.0, zero_division=0)
        brier = brier_score_loss(y_test, probs_test)

        results.append({
            'Horizon': h_name,
            'Model': m_name,
            'ROC-AUC': round(roc_auc, 4),
            'PR-AUC': round(pr_auc, 4),
            'Accuracy': round(acc, 4),
            'Optimal Thresh': round(opt_th, 2),
            'Sensitivity (Recall)': round(rec, 4),
            'Precision': round(prec, 4),
            'F1-Score': round(f1_opt, 4),
            'F2-Score (Public Health)': round(f2_opt, 4),
            'Brier Score': round(brier, 4)
        })

df_benchmarks = pd.DataFrame(results)
df_benchmarks.to_csv(PROCESSED_DATA_DIR / 'cchain_model_benchmarks.csv', index=False)
display(df_benchmarks.sort_values(by=['Horizon', 'PR-AUC'], ascending=[True, False]))

## Step 6: Publication-Grade Model Evaluation Visualizations

We generate **ROC Curves**, **Precision-Recall Curves (PR-AUC)**, and **Feature Importance Plots** to evaluate diagnostic discrimination on the holdout test partition.

In [ ]:
# Plot 1 & 2: ROC Curves & Precision-Recall Curves for 30-Day Lead Horizon
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
model_names = ['Logistic Regression', 'Random Forest', 'LightGBM', 'XGBoost']
colors = {'Logistic Regression': '#7570b3', 'Random Forest': '#1b9e77', 'LightGBM': '#d95f02', 'XGBoost': '#e7298a'}

for m_name in model_names:
    key = f'{m_name}_30-Day Lead (T+1)'
    probs = predictions_dict[key]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    ax1.plot(fpr, tpr, label=f'{m_name} (AUC = {auc_val:.3f})', color=colors[m_name], linewidth=2)

    # PR Curve
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc_val = average_precision_score(y_test, probs)
    ax2.plot(rec, prec, label=f'{m_name} (PR-AUC = {pr_auc_val:.3f})', color=colors[m_name], linewidth=2)

# Formatting ROC
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontweight='bold')
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontweight='bold')
ax1.set_title('ROC Curves (30-Day Early Warning Horizon)', fontsize=12, fontweight='bold')
ax1.legend(loc='lower right', frameon=True)

# Formatting PR Curve
base_rate = y_test.mean()
ax2.axhline(base_rate, color='k', linestyle='--', linewidth=1, label=f'Baseline ({base_rate:.2f})')
ax2.set_xlabel('Recall (Sensitivity)', fontweight='bold')
ax2.set_ylabel('Precision', fontweight='bold')
ax2.set_title('Precision-Recall Curves (30-Day Early Warning Horizon)', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Top Feature Importances (Random Forest / LightGBM)
key_30d = 'Random Forest_30-Day Lead (T+1)'
rf_model = trained_models_dict[key_30d]
feat_list = feature_cols_dict[key_30d]
imps = pd.Series(rf_model.feature_importances_, index=feat_list).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6.5))
imps.tail(12).plot(kind='barh', ax=ax, color='#2c7fb8', edgecolor='black')
ax.set_title('Top 12 Most Predictive Drivers: 30-Day Dengue Outbreak Surveillance', fontsize=12, fontweight='bold')
ax.set_xlabel('Relative Gini Feature Importance Weight', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7: Prescriptive LGU Action Framework & Decision Matrix

The surveillance engine maps model output probabilities into an **Automated 3-Tier LGU Decision-Support Matrix**:

```
┌─────────────────────────────────────────────────────────────────────────────────────────────┐
│ Probability (P)  │ Alert Level     │ Automated Prescriptive Response                        │
├──────────────────┼─────────────────┼────────────────────────────────────────────────────────┤
│ P < 0.30         │ Level 1: Normal │ Standard community cleanup & entomological monitoring. │
│ 0.30 <= P < 0.65 │ Level 2: Alert  │ Pre-emptive larviciding in high-density container zones│
│                  │                 │ and dispatch community health workers (CHWs).          │
│ P >= 0.65        │ Level 3: Outbreak│ Targeted spatial fogging within 48h; triage rehydration│
│                  │ Warning         │ tents; pre-allocate 200 rapid dengue NS1 test kits.    │
└─────────────────────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Generate Sample Real-Time LGU Dispatch Schedule (Latest Test Month)
latest_test_date = df_final.loc[test_mask, 'date'].max()
df_latest = df_final[df_final['date'] == latest_test_date].copy()

best_model = trained_models_dict['Random Forest_30-Day Lead (T+1)']
df_latest['outbreak_prob'] = best_model.predict_proba(df_latest[FEATURES_30D_HORIZON].fillna(0))[:, 1]

def assign_alert_tier(p):
    if p >= 0.65:
        return 'LEVEL 3: CRITICAL OUTBREAK WARNING'
    elif p >= 0.30:
        return 'LEVEL 2: PRE-EMPTIVE ALERT'
    else:
        return 'LEVEL 1: NORMAL MONITORING'

df_latest['Alert_Status'] = df_latest['outbreak_prob'].apply(assign_alert_tier)

lgu_dispatch_table = df_latest[[
    'adm4_en', 'pop_count_total', 'google_bldgs_pct_built_up_area', 
    'pr_total_mm_lag_2m', 'heat_index_mean_lag_2m', 'outbreak_prob', 'Alert_Status'
]].sort_values(by='outbreak_prob', ascending=False)

print(f'[+] Automated Prescriptive LGU Action Table for Forecast Period: {latest_test_date.strftime("%B %Y")}')
display(lgu_dispatch_table.head(10))